# RG3 F1: 전문가 점수 단위 안정화

94번 모델의 저장된 학습 ref와 점수만 재사용. 원래점수/학습 정상95분위수 나눔/학습 정상 logit 중앙값-IQR 조절 세 후보. 각 ref의 학습 정상만 기준 계산. 모델 학습 정상은 in-sample이므로 확률 calibration이라고 부르지 않음. 외부 임계값/가중치/방식 선택은 외부 제외 내부 OOF 점수 및 라벨만 이용. primary는 inner_F1_selected_calibration_and_blend.

고정 RG3 RH620·가스19·정상601. 새 독립 검증 아닌 개발용 중첩검증. 저장된 모델 ref에 평가 구간이 없는지 모든 조회마다 assert. 변환센서 cache의 라벨을 전역피처/임계값 학습에 사용하지 않음.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from scipy.special import logit
from sklearn.metrics import confusion_matrix,precision_recall_curve,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
z=np.load(t/'rg3_F1_boost_fold_cache.npz',allow_pickle=False)
y=z['labels'];groups=z['groups'];times=z['timestamps'];blocks=np.unique(groups)
assert len(y)==620 and y.sum()==19
cache={}
for key in z.files:
 if key.startswith('ref_'):
  n=key.split('_')[1];ref=z[key].astype(int);raw=z['scores_'+n]
  assert raw.shape==(620,6)
  normal=ref[y[ref]==0]
  unit=np.maximum(np.quantile(raw[normal],.95,axis=0),.01)
  logits=logit(np.clip(raw,1e-4,1-1e-4))
  loc=np.median(logits[normal],axis=0)
  scale=np.maximum(np.quantile(logits[normal],.75,axis=0)-np.quantile(logits[normal],.25,axis=0),.25)
  cache[tuple(ref)]={'raw':raw,'normal95_scaled':raw/unit,'normal_logit_scaled':(logits-loc)/scale}
def get(ref,q,method):
 assert set(groups[ref]).isdisjoint(groups[q])
 return cache[tuple(ref)][method][q]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,recall=tp/(tp+fn) if tp+fn else 0)
def cutfit(yy,s):
 p,r,c=precision_recall_curve(yy,s)
 f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(c),where=p[:-1]+r[:-1]>0)
 order=np.lexsort((c,r[:-1],f));cut=float(c[order[-1]])
 return cut,stats(yy,s>=cut)
cols=[0,1,2,4,5]
weights=[np.eye(5)[i] for i in range(5)]+[(np.eye(5)[i]+np.eye(5)[j])/2 for i in range(5) for j in range(i+1,5)]
records=[];choices=[];search=[]
for b in blocks:
 q=np.flatnonzero(groups==b);ref=np.flatnonzero(groups!=b)
 candidates=[];method_winners=[]
 for method in ['raw','normal95_scaled','normal_logit_scaled']:
  out=get(ref,q,method);inner=np.full((len(ref),6),np.nan)
  for ib in np.unique(groups[ref]):
   mask=groups[ref]==ib;inner[mask]=get(ref[~mask],ref[mask],method)
  local=[]
  for w in weights:
   cut,result=cutfit(y[ref],inner[:,cols]@w);score=out[:,cols]@w
   entry=(result['F1'],result['recall'],-result['FP'],method,w,cut,score)
   local.append(entry);candidates.append(entry)
   search.append(dict(block=int(b),method=method,weights=','.join(map(str,w)),threshold=cut,**result))
  win=max(local,key=lambda a:a[:3]);method_winners.append(win)
 best=max(candidates,key=lambda a:a[:3])
 named=[(win,win[3]+'_inner_F1_blend') for win in method_winners]+[(best,'inner_F1_selected_calibration_and_blend')]
 for win,name in named:
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=times[i],block=int(b),actual=int(y[i]),prediction=int(win[6][j]>=win[5]),score=float(win[6][j]),threshold=win[5]))
 choices.append(dict(block=int(b),method=best[3],weights=','.join(map(str,best[4])),threshold=best[5],inner_F1=best[0],inner_recall=best[1]))
 print('Finished calibrated F1 block',b,flush=True)
# Selected winner may also be a local winner; retain separately named procedures.
# Each model/shot pair must occur exactly once.
p=pd.DataFrame(records)
assert not p.duplicated(['model','TimeStamp']).any()
rows=[]
for name,g in p.groupby('model'):
 assert len(g)==620
 rows.append(dict(model=name,AP=average_precision_score(g.actual,g.score),**stats(g.actual,g.prediction)))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(search),'inner_search')]:obj.to_csv(t/f'rg3_F1_calibrated_blend_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))


Finished calibrated F1 block 1


Finished calibrated F1 block 2


Finished calibrated F1 block 3


Finished calibrated F1 block 4


Finished calibrated F1 block 5


Finished calibrated F1 block 6


Finished calibrated F1 block 7


                                  model       AP  TP  FP  FN  TN       F1   recall
inner_F1_selected_calibration_and_blend 0.101266   3  30  16 571 0.115385 0.157895
         normal95_scaled_inner_F1_blend 0.072886   1  14  18 587 0.058824 0.052632
     normal_logit_scaled_inner_F1_blend 0.070590   3  46  16 555 0.088235 0.157895
                     raw_inner_F1_blend 0.069945   3  19  16 582 0.146341 0.157895
